In [0]:
#adding src path and reading config file
import sys
sys.path.append("/Workspace/DataStore/DataStore/src")
import yaml
from core.Base import BaseConfig
from core.Read_Yaml import ReadYaml
from core.Create_Tabls import CreateTable
from core.Create_Schem import CreateSchema
from core.TableHandler import TableHandler
from pyspark.sql.functions import current_timestamp
sys.path.append("/Workspace/DataStore/DataStore/src")
path = "/Workspace/DataStore/DataStore/configs/bronze.yaml"
settings_path = "/Workspace/DataStore/DataStore/configs/settings.yaml"


In [0]:
#getting table information and creating table and schema if not exists
table_info= ReadYaml.read_yaml(path, "User")
table_info = BaseConfig(table_info)
config = {
    "spark":spark,  
    "catalog": table_info.catalog,
    "schema":table_info.schema,
    "table_name":table_info.tableName,
    "path":table_info.path,
    "table_type":table_info.table_type
}
CreateSchema.create_schema(spark,table_info.catalog, table_info.schema)
CreateTable.create_table(config=config)

In [0]:
#reading data from source path
source_path = ReadYaml.read_yaml(settings_path, "paths")
source_user = f"{source_path['source']}/{table_info.tableName}"
schema_location = "abfss://bronze@datastorestorageaccount.dfs.core.windows.net/Z_Schema/User/"
checkpoint_path = "abfss://bronze@datastorestorageaccount.dfs.core.windows.net/Z_Checkpoint/User/"

# user_df = spark.read.parquet(source_user)
# user_df = user_df.withColumn("InsertedAt", current_timestamp())

# #writing data to the bronze external table
# TableHandler.write(df= user_df, spark=spark,
#                     table_name=f"{table_info.catalog}.{table_info.schema}.{table_info.tableName}",
#                      mergeSchema=True,
#                       mode="overwrite")



In [0]:
def transform(df):
    return (
        df.withColumn("InsertedAt", current_timestamp())
    ) 
       

TableHandler.readWriteStream(spark=spark,
                     tableName=f"{table_info.catalog}.{table_info.schema}.{table_info.tableName}",
                     source_path=source_user,
                     schema_location=schema_location,
                     checkpoint_path=checkpoint_path,
                     file_type="parquet",
                     schemaEvolutionMode="addNewColumns",
                     useNotifications=True,
                     inferColumnTypes=True,
                     write_mode="append",
                     transform_fn=transform)
    


In [0]:
# spark.sql("update datastore.bronze.User set name = 'Adam Leelala' where id = '266d9438-f297-41dc-b319-cf8861e8eccd'")

In [0]:
%sql
select count(1) from datastore.gold.user --where ExternalId = "266d9438-f297-41dc-b319-cf8861e8eccd";


In [0]:
%sql
select * from datastore.bronze.user